# Khmer Query Correction Test

Use `test_query()` with Khmer text directly in a notebook cell. The diagnostic output helps identify font/rendering problems separately from Unicode normalization and spelling correction.

In [8]:
import sys
from pathlib import Path

DATA_PIPELINE = None
for path in (Path.cwd(), *Path.cwd().parents):
    if path.name == 'data-pipeline':
        DATA_PIPELINE = path
        break
    if (path / 'data-pipeline').is_dir():
        DATA_PIPELINE = path / 'data-pipeline'
        break
if DATA_PIPELINE is None:
    raise FileNotFoundError('Could not locate the data-pipeline directory')
if str(DATA_PIPELINE) not in sys.path:
    sys.path.insert(0, str(DATA_PIPELINE))

from dense_retrieval.query_correction import correct_query

In [9]:
def test_query(query: str) -> dict:
    """Print and return normalization/correction results for one query."""
    result = correct_query(query)
    output = {
        'original': result.original,
        'normalized': result.normalized,
        'corrected': result.corrected,
        'suggestions': result.suggestions,
        'variants': result.variants,
        'original_codepoints': [f'U+{ord(char):04X}' for char in result.original],
        'normalized_codepoints': [f'U+{ord(char):04X}' for char in result.normalized],
        'corrected_codepoints': [f'U+{ord(char):04X}' for char in result.corrected],
    }
    for key, value in output.items():
        print(f'{key}: {value}')
    return output

## Test a Khmer typo

Edit the text inside `test_query(...)` and run the cell.

In [10]:
test_query('សម្បត្ត')

original: សម្បត្ត
normalized: សម្បត្ត
corrected: សម្បត្តិ
suggestions: ('សម្បត្តិ',)
variants: ('សម្បត្ត', 'សម្បត្តិ')
original_codepoints: ['U+179F', 'U+1798', 'U+17D2', 'U+1794', 'U+178F', 'U+17D2', 'U+178F']
normalized_codepoints: ['U+179F', 'U+1798', 'U+17D2', 'U+1794', 'U+178F', 'U+17D2', 'U+178F']
corrected_codepoints: ['U+179F', 'U+1798', 'U+17D2', 'U+1794', 'U+178F', 'U+17D2', 'U+178F', 'U+17B7']


{'original': 'សម្បត្ត',
 'normalized': 'សម្បត្ត',
 'corrected': 'សម្បត្តិ',
 'suggestions': ('សម្បត្តិ',),
 'variants': ('សម្បត្ត', 'សម្បត្តិ'),
 'original_codepoints': ['U+179F',
  'U+1798',
  'U+17D2',
  'U+1794',
  'U+178F',
  'U+17D2',
  'U+178F'],
 'normalized_codepoints': ['U+179F',
  'U+1798',
  'U+17D2',
  'U+1794',
  'U+178F',
  'U+17D2',
  'U+178F'],
 'corrected_codepoints': ['U+179F',
  'U+1798',
  'U+17D2',
  'U+1794',
  'U+178F',
  'U+17D2',
  'U+178F',
  'U+17B7']}

## Test several queries

In [11]:
queries = [
    'សម្បត្ត',
    'សួរស្តី',
    'ភាសា',
    '\u200bសួរស្តី',
]

for query in queries:
    print('---')
    test_query(query)

---
original: សម្បត្ត
normalized: សម្បត្ត
corrected: សម្បត្តិ
suggestions: ('សម្បត្តិ',)
variants: ('សម្បត្ត', 'សម្បត្តិ')
original_codepoints: ['U+179F', 'U+1798', 'U+17D2', 'U+1794', 'U+178F', 'U+17D2', 'U+178F']
normalized_codepoints: ['U+179F', 'U+1798', 'U+17D2', 'U+1794', 'U+178F', 'U+17D2', 'U+178F']
corrected_codepoints: ['U+179F', 'U+1798', 'U+17D2', 'U+1794', 'U+178F', 'U+17D2', 'U+178F', 'U+17B7']
---
original: សួរស្តី
normalized: សួរស្តី
corrected: សួស្ដី
suggestions: ('សួស្ដី',)
variants: ('សួរស្តី', 'សួស្ដី')
original_codepoints: ['U+179F', 'U+17BD', 'U+179A', 'U+179F', 'U+17D2', 'U+178F', 'U+17B8']
normalized_codepoints: ['U+179F', 'U+17BD', 'U+179A', 'U+179F', 'U+17D2', 'U+178F', 'U+17B8']
corrected_codepoints: ['U+179F', 'U+17BD', 'U+179F', 'U+17D2', 'U+178A', 'U+17B8']
---
original: ភាសា
normalized: ភាសា
corrected: ភាសា
suggestions: ()
variants: ('ភាសា',)
original_codepoints: ['U+1797', 'U+17B6', 'U+179F', 'U+17B6']
normalized_codepoints: ['U+1797', 'U+17B6', 'U+179F'

## Legal spelling benchmark

This cell tests the legal terms and full Khmer queries provided for LegalRAG.

In [12]:
legal_cases = {
    'Legal terminology': [
        ('ចបាប់', 'ច្បាប់'),
        ('រដធម្មនុញ្ញ', 'រដ្ឋធម្មនុញ្ញ'),
        ('តុលាកា', 'តុលាការ'),
        ('បទល្មើ', 'បទល្មើស'),
        ('ព្រហ្មទន្ឌ', 'ព្រហ្មទណ្ឌ'),
        ('ទន្ឌកម្ម', 'ទណ្ឌកម្ម'),
        ('សិទិ្ធ', 'សិទ្ធិ'),
        ('កាតពកិច្ច', 'កាតព្វកិច្ច'),
        ('យុតិធម៌', 'យុត្តិធម៌'),
        ('បណ្តឹង', 'បណ្ដឹង'),
        ('កិចសន្យា', 'កិច្ចសន្យា'),
    ],
    'Government and administration': [
        ('រដាភិបាល', 'រដ្ឋាភិបាល'),
        ('ក្រសួ', 'ក្រសួង'),
        ('អាជ្ញាធ', 'អាជ្ញាធរ'),
        ('រដបាល', 'រដ្ឋបាល'),
        ('មន្រ្តី', 'មន្ត្រី'),
        ('ប្រជាធិបតេយ', 'ប្រជាធិបតេយ្យ'),
        ('សាធារណ', 'សាធារណៈ'),
        ('គោលនយោបា', 'គោលនយោបាយ'),
        ('អនុវត', 'អនុវត្ត'),
    ],
    'Social and traffic terms': [
        ('ចរាចរ', 'ចរាចរណ៍'),
        ('គ្រោះថ្នា', 'គ្រោះថ្នាក់'),
        ('សុវត្តិភាព', 'សុវត្ថិភាព'),
        ('គ្រឿងញៀ', 'គ្រឿងញៀន'),
        ('អំពើហិង្សា', 'អំពើហិង្សា'),
        ('អប់រំ', 'អប់រំ'),
        ('បរិស្ថា', 'បរិស្ថាន'),
        ('សង្គមកិច', 'សង្គមកិច្ច'),
        ('សេដកិច្ច', 'សេដ្ឋកិច្ច'),
    ],
    'Full Khmer queries': [
        ('តើការបើកបរមិនពាក់មួកសុវត្តិភាពខុសចបាប់ទេ', 'តើការបើកបរមិនពាក់មួកសុវត្ថិភាពខុសច្បាប់ទេ'),
        ('តើបទល្មើចរាចរមានទន្ឌកម្មអ្វីខ្លះ', 'តើបទល្មើសចរាចរណ៍មានទណ្ឌកម្មអ្វីខ្លះ'),
        ('តើកម្មករមានសិទិ្ធអ្វីខ្លះ', 'តើកម្មករមានសិទ្ធិអ្វីខ្លះ'),
        ('តើអាចដាក់បណ្តឹងទៅតុលាកាបានទេ', 'តើអាចដាក់បណ្ដឹងទៅតុលាការបានទេ'),
        ('តើកិចសន្យាការងារមានកាតពកិច្ចអ្វីខ្លះ', 'តើកិច្ចសន្យាការងារមានកាតព្វកិច្ចអ្វីខ្លះ'),
    ],
}

for category, cases in legal_cases.items():
    print(f'\n### {category}')
    for wrong, expected in cases:
        result = correct_query(wrong)
        status = 'PASS' if result.corrected == expected else 'CHECK'
        print(f'[{status}]')
        print(f'  Input:    {wrong}')
        print(f'  Result:   {result.corrected}')
        print(f'  Expected: {expected}')
        print(f'  Variants: {result.variants}')


### Legal terminology
[CHECK]
  Input:    ចបាប់
  Result:   ចំបាប់
  Expected: ច្បាប់
  Variants: ('ចបាប់', 'ចំបាប់')
[CHECK]
  Input:    រដធម្មនុញ្ញ
  Result:   រដធម្មនុញ្ញ
  Expected: រដ្ឋធម្មនុញ្ញ
  Variants: ('រដធម្មនុញ្ញ',)
[PASS]
  Input:    តុលាកា
  Result:   តុលាការ
  Expected: តុលាការ
  Variants: ('តុលាកា', 'តុលាការ')
[CHECK]
  Input:    បទល្មើ
  Result:   បទល្មើ
  Expected: បទល្មើស
  Variants: ('បទល្មើ',)
[CHECK]
  Input:    ព្រហ្មទន្ឌ
  Result:   ព្រហ្មទន្ឌ
  Expected: ព្រហ្មទណ្ឌ
  Variants: ('ព្រហ្មទន្ឌ',)
[CHECK]
  Input:    ទន្ឌកម្ម
  Result:   ទន្ឌកម្ម
  Expected: ទណ្ឌកម្ម
  Variants: ('ទន្ឌកម្ម',)
[CHECK]
  Input:    សិទិ្ធ
  Result:   សិទិ្ធ
  Expected: សិទ្ធិ
  Variants: ('សិទិ្ធ',)
[CHECK]
  Input:    កាតពកិច្ច
  Result:   កាតពកិច្ច
  Expected: កាតព្វកិច្ច
  Variants: ('កាតពកិច្ច',)
[CHECK]
  Input:    យុតិធម៌
  Result:   យុតិធម៌
  Expected: យុត្តិធម៌
  Variants: ('យុតិធម៌',)
[CHECK]
  Input:    បណ្តឹង
  Result:   បណ្តឹង
  Expected: បណ្ដឹង
  Variants: ('បណ្តឹង',)
[C

## Font diagnosis

If the Khmer text looks incorrect but the code points are valid, the problem is probably the notebook/editor font. If code points differ unexpectedly, the problem is Unicode input or normalization.

In [13]:
from IPython.display import display, HTML

display(HTML('''
<div style="font-family: 'Noto Sans Khmer', 'Khmer OS', sans-serif; font-size: 28px; line-height: 1.8;">
  សម្បត្តិ · សួស្ដី · ប័ណ្ណបើកបរ · ច្បាប់ចរាចរណ៍
</div>
'''))